# 08 — Attn-LOB LOB-Depth Ablation (Exp 3)

Exp 2 found Attn-LOB's ordering completely inverted from the paper's Table I. The natural follow-up: can we close the gap with **more data**? We checked `rds_transformed.parquet` (and the R2 bucket it came from) as a source of additional BTC/USDT history for Attn-LOB specifically.

That investigation turned up a dead end: `rds_transformed.parquet` covers only 2022-10-20 (8,127 snapshots), and that day is already the *first* day inside `btc_usdt_20221019_20221030_lob.parquet` — the file every prior experiment already trains on. We confirmed this directly: `BTCUSDT.ask_price_1..10` in the existing multi-day file matches `rds_transformed.parquet`'s best bid/ask, row for row, for all 8,127 rows on that day. There is no new *time* in this data.

What `rds_transformed.parquet` *does* have that we've never used: full order-book depth. Our pipeline has always truncated each snapshot to the top 10 price levels per side; the raw file carries thousands (mean ~11,400 ask levels, ~9,300 bid levels) at the same ~10s cadence. So instead of *more data*, this experiment tests **more depth**: does giving Attn-LOB the top 50 levels instead of top 10, on the exact same single day, same samples, same split, same hyperparameters, change anything?

This is a controlled ablation, not a data-volume experiment — both arms use identical rows (8,068 windowed samples from 2022-10-20 only), so any difference is attributable to depth alone. It's also a much *smaller* dataset than Exp 1/1b/2 (one day vs. eleven), so absolute numbers here aren't directly comparable to those experiments — only the two arms within this notebook are comparable to each other.

In [ ]:
from __future__ import annotations

import pathlib

import pandas as pd
import polars as pl
import torch
from torch.utils.data import DataLoader

from paper_replication.features import FeatureConfig, build_feature_dataset
from paper_replication.features.dataset import chronological_split
from paper_replication.models import AttnLOB, AttnLOBConfig
from paper_replication.training import TrainConfig, evaluate, train_model
from paper_replication.training.dataset import AttnLOBTorchDataset

RDS_PATH = "../data/rds_transformed.parquet"
EXISTING_LOB_PATH = "../data/btc_usdt_20221019_20221030_lob.parquet"
PROCESSED_DIR = pathlib.Path("../data/processed")
SYMBOL = "BTCUSDT"

torch.manual_seed(0)

## Build two flattened snapshot files from the same raw depth source

`rds_transformed.parquet` stores each snapshot's `asks`/`bids` as a list of `{price, size}` structs, already sorted (ascending for asks, descending for bids — verified across all rows). We flatten the top `n_levels` of each into the same `SYMBOL.ask_price_i` / `SYMBOL.ask_size_i` / ... column layout `lob_state.py` expects, for `n_levels=10` (matching every prior experiment) and `n_levels=50` (the depth arm).

The dynamic-state pipeline (`dynamic_state.py`) also needs `mid_price`, `net_trade_sign`, `trade_count`, `ema_ofi` — none of which exist in the raw depth file. Rather than re-deriving them, we copy them straight from the existing multi-day file's 2022-10-20 rows: we already confirmed those rows correspond 1:1 (by sorted position) with `rds_transformed.parquet`'s rows, so this is exactly the value the existing pipeline would have computed, not an approximation.

In [ ]:
existing = pd.read_parquet(
    EXISTING_LOB_PATH,
    columns=["timestamp", "mid_price", "net_trade_sign", "trade_count", "ema_ofi"],
)
ts = pd.to_datetime(existing["timestamp"], unit="s")
oct20_mask = (ts >= "2022-10-20") & (ts < "2022-10-21")
extra_cols = existing[oct20_mask].sort_values("timestamp").reset_index(drop=True)
print(f"extra_cols rows: {len(extra_cols)}")


def build_flat_snapshot(n_levels: int) -> pd.DataFrame:
    lazy = pl.scan_parquet(RDS_PATH).select("time", "asks", "bids").sort("time")
    exprs = []
    for i in range(n_levels):
        exprs.append(
            pl.col("asks")
            .list.get(i)
            .struct.field("price")
            .alias(f"{SYMBOL}.ask_price_{i+1}")
        )
        exprs.append(
            pl.col("asks")
            .list.get(i)
            .struct.field("size")
            .alias(f"{SYMBOL}.ask_size_{i+1}")
        )
        exprs.append(
            pl.col("bids")
            .list.get(i)
            .struct.field("price")
            .alias(f"{SYMBOL}.bid_price_{i+1}")
        )
        exprs.append(
            pl.col("bids")
            .list.get(i)
            .struct.field("size")
            .alias(f"{SYMBOL}.bid_size_{i+1}")
        )
    flat = lazy.select(exprs).collect().to_pandas()
    assert len(flat) == len(extra_cols)
    out = pd.concat([extra_cols, flat], axis=1)
    assert out.isnull().sum().sum() == 0
    return out


PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DEPTH_PATHS = {}
for n_levels in (10, 50):
    df = build_flat_snapshot(n_levels)
    path = PROCESSED_DIR / f"btc_usdt_20221020_lob_n{n_levels}.parquet"
    df.to_parquet(path)
    DEPTH_PATHS[n_levels] = str(path)
    print(f"n_levels={n_levels}: shape={df.shape} -> {path}")

## Build the feature datasets and split

Same `FeatureConfig` defaults as every prior experiment (`window_T=50`, `horizon_k=10`, auto-calibrated label `alpha`), varying only `n_levels`. Because both arms come from the exact same 8,127 raw snapshots, the sample counts, split sizes, and label balance should come out identical — confirming the only thing that differs between arms is depth.

In [ ]:
arms = {}
for n_levels, snapshot_path in DEPTH_PATHS.items():
    feature_config = FeatureConfig(n_levels=n_levels)
    dataset = build_feature_dataset(snapshot_path, feature_config)
    splits = chronological_split(dataset)
    arms[n_levels] = {
        "feature_config": feature_config,
        "dataset": dataset,
        "splits": splits,
    }
    print(
        f"n_levels={n_levels:>2}: total={len(dataset):>5}  lob_state.shape={dataset.lob_state.shape}  "
        f"train={len(splits['train'])} val={len(splits['val'])} test={len(splits['test'])}  "
        f"alpha={dataset.alpha_used:.6g}"
    )

## Train Attn-LOB at n_levels=10 vs. n_levels=50

Both arms use Exp 1b's winning hyperparameters (`dropout=0.3`, everything else `TrainConfig`'s untuned defaults: `lr=1e-3`, `weight_decay=0.0`, `batch_size=256`, `patience=5`, `seed=0`) — the best-known Attn-LOB config so far, held fixed across arms so `n_levels` is the only variable. `ConvBlock`'s width-collapsing conv stack (`layers.py`) now generalizes beyond the paper's fixed `n_levels=10` case (previously a hard `NotImplementedError`) precisely to make this ablation possible — `n_levels=50` is a deviation from the paper's Fig. 1, not a claim it reflects their architecture.

In [ ]:
for n_levels, arm in arms.items():
    model = AttnLOB(AttnLOBConfig(n_levels=n_levels, dropout=0.3))
    n_params = sum(p.numel() for p in model.parameters())
    checkpoint_path = PROCESSED_DIR / f"attn_lob_depth_n{n_levels}_checkpoint.pt"
    train_config = TrainConfig(
        batch_size=256,
        learning_rate=1e-3,
        seed=0,
        checkpoint_path=str(checkpoint_path),
    )

    history = train_model(
        model, arm["splits"]["train"], arm["splits"]["val"], train_config, num_classes=3
    )
    best_epoch = min(history, key=lambda r: r["val_loss"])

    arm["model"] = model
    arm["train_config"] = train_config
    arm["history"] = history
    arm["best_epoch"] = best_epoch
    arm["n_params"] = n_params
    print(
        f"n_levels={n_levels:>2}: {n_params:>7,} params  ran {len(history):2d} epochs  "
        f"best_val_loss={best_epoch['val_loss']:.4f}  best_val_f1={best_epoch['f1']:.4f}"
    )

## Evaluate each arm on its held-out test split — exactly once

In [ ]:
for n_levels, arm in arms.items():
    device = torch.device(arm["train_config"].device)
    loader = DataLoader(
        AttnLOBTorchDataset(arm["splits"]["test"]), batch_size=256, shuffle=False
    )
    metrics = evaluate(arm["model"], loader, device, num_classes=3)
    arm["test_metrics"] = metrics
    print(
        f"n_levels={n_levels:>2}: precision={metrics['precision']:.4f}  "
        f"recall={metrics['recall']:.4f}  f1={metrics['f1']:.4f}  accuracy={metrics['accuracy']:.4f}"
    )

## Compare depth=10 vs. depth=50, and both against the multi-day Exp 1b baseline

The Exp 1b row below is *not* a controlled comparison — it trains on eleven days (35,467 train samples) instead of one (3,227 here) — it's included only as context for how much of a gap sample count alone is likely responsible for.

In [ ]:
EXP_1B_MULTI_DAY_F1 = 0.5224  # notebooks/06, 35,467 train samples across 11 days

print(
    f"{'Arm':>28} {'Train n':>8} {'Precision':>10} {'Recall':>8} {'F1':>8} {'Accuracy':>9}"
)
for n_levels, arm in arms.items():
    m = arm["test_metrics"]
    train_n = len(arm["splits"]["train"])
    print(
        f"{'n_levels=' + str(n_levels) + ' (Oct-20 only)':>28} {train_n:>8} "
        f"{m['precision']:>10.4f} {m['recall']:>8.4f} {m['f1']:>8.4f} {m['accuracy']:>9.4f}"
    )
print(
    f"{'Exp 1b, n_levels=10 (11 days)':>28} {35467:>8} {'--':>10} {'--':>8} {EXP_1B_MULTI_DAY_F1:>8.4f} {'--':>9}"
)

delta_f1 = arms[50]["test_metrics"]["f1"] - arms[10]["test_metrics"]["f1"]
print(
    f"\ndepth-50 vs. depth-10 F1 delta (same day, same samples, same hyperparameters): {delta_f1:+.4f}"
)

## Summary

**Depth helped, modestly.** Going from `n_levels=10` to `n_levels=50` on the exact same 8,068 samples, split, and hyperparameters moved test F1 from 0.4411 to 0.4679 (+0.0269, ~6% relative) and best val_loss from 1.1104 to 1.0444 — a real, consistent improvement across both the selection metric (val_loss) and the held-out metric (F1), not just noise in one direction. Precision was essentially flat (0.5072 vs 0.5073); the gain came entirely from recall (0.4543 -> 0.4697) and accuracy (0.4487 -> 0.4695).

**It doesn't come close to closing the gap that matters.** Both depth arms — trained on one day (3,227 train samples) — land well below Exp 1b's F1=0.5224, which used the same `n_levels=10` architecture but eleven days (35,467 train samples). Depth bought six points of relative improvement; the missing ten days cost far more than that. On this dataset, sample count dominates depth by a wide margin.

**What this does and doesn't show:**

- **The original "more data" question is answered, just not the way we expected.** There is no more BTC/USDT LOB history available in this repo or the R2 bucket beyond the eleven days already used — `rds_transformed.parquet` is a strict subset (a single already-included day) with extra depth, not extra time. This ablation is the closest available proxy: "does giving the model more information about that data help," and the answer is yes, but only a little.
- **Not a claim that n_levels=50 is a better architecture.** It's a deliberate deviation from the paper's Fig. 1 (n_levels=10), enabled here only to test this specific question. `AttnLOBConfig(n_levels=50)` isn't used anywhere else in this replication.
- **Single split, single seed.** Like Exp 1/1b/2, this is one train/val/test split and one run per arm — a real effect given the consistent direction across both val_loss and test F1, but not error-barred.
- **Natural next step, if it's worth the compute:** repeat with an intermediate depth (e.g. `n_levels=20` or `30`) to see whether the F1 gain scales roughly linearly with depth or saturates quickly — would clarify whether depth=50 is already past the point of diminishing returns for this asset's order-book structure.